In [ ]:
import os
import uuid

from dotenv import load_dotenv

# Load .env from this agent's directory
load_dotenv(override=True)

from react_with_database_memory.agent import get_graph_closure
from react_with_database_memory.utils import get_database_uri

# Build DB URI from env vars (handles password URL-encoding)
DB_URI = get_database_uri()

# Config
SYSTEM_PROMPT = (
    "You are a helpful AI assistant. Answer questions concisely in 1-2 sentences."
)
THREAD_ID = f"fifo-test-{uuid.uuid4().hex[:8]}"

print(f"Model:     {os.getenv('MODEL_ID')}")
print(f"LLM:       {os.getenv('BASE_URL')}")
print(f"Thread ID: {THREAD_ID}")

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.postgres import PostgresSaver

# Get graph closure (reads MODEL_ID, BASE_URL, API_KEY from env)
agent_closure = get_graph_closure()

# Init DB tables
with PostgresSaver.from_conn_string(DB_URI) as saver:
    saver.setup()
    print("PostgreSQL checkpoint tables ready")

print("Agent closure ready")

In [ ]:
def chat(message: str, thread_id: str = None):
    """Send a message to the agent. Messages persist in PostgreSQL across calls."""
    tid = thread_id or THREAD_ID
    config = {"configurable": {"thread_id": tid}}

    with PostgresSaver.from_conn_string(DB_URI) as saver:
        agent = agent_closure(saver, tid, SYSTEM_PROMPT)
        result = agent.invoke(
            {"messages": [HumanMessage(content=message)]}, config=config
        )

    total = len(result["messages"])
    response = result["messages"][-1].content
    print(f"  [{total} messages total in PostgreSQL state]")
    print(f"  You:   {message}")
    print(f"  Agent: {response}")
    print()
    return result


print("chat() function ready")

## FIFO Memory Demo

With `MAX_MESSAGES_IN_CONTEXT = 5`, the FIFO window works like this:

| Turn | State msgs | LLM sees | Jonny visible? |
|------|-----------|----------|----------------|
| 1    | H1        | 1        | YES |
| 2    | H1,A1,H2  | 3        | YES |
| 3    | 6 msgs    | 5        | YES (barely) |
| 4    | 8 msgs    | 5        | maybe (A2 ref) |
| 5    | 10 msgs   | 5        | **NO** |

In [9]:
# Turn 1: Tell a story about Jonny
print("=== Turn 1: Tell the Jonny story ===")
chat(
    "Let me tell you about Jonny. He is a lovely boy with bright blue eyes "
    "and curly red hair. He loves playing guitar and eating pizza."
)

{'messages': [HumanMessage(content='Let me tell you about Jonny. He is a lovely boy with bright blue eyes and curly red hair. He loves playing guitar and eating pizza.', additional_kwargs={}, response_metadata={}, id='18714510-602c-48cb-9194-8741467a1614'),
  AIMessage(content="That sounds like a charming young person! It's great that he has found interests that bring him joy, such as music and food.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 74, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama/llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-72', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cade9-6c32-76c3-bd0c-f71c63d6868d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 28, 'total_tokens': 102, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
# Turn 2: Verify the agent remembers (H1 still in context)
print("=== Turn 2: Agent should REMEMBER Jonny ===")
chat("What color are Jonny's eyes?")

{'messages': [HumanMessage(content='Let me tell you about Jonny. He is a lovely boy with bright blue eyes and curly red hair. He loves playing guitar and eating pizza.', additional_kwargs={}, response_metadata={}, id='18714510-602c-48cb-9194-8741467a1614'),
  AIMessage(content="That sounds like a charming young person! It's great that he has found interests that bring him joy, such as music and food.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 74, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama/llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-72', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cade9-6c32-76c3-bd0c-f71c63d6868d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 28, 'total_tokens': 102, 'input_token_details': {}, 'output_token_details': {}})

In [6]:
# Turn 3: Filler message
print("=== Turn 3: Filler ===")
chat("What is the capital of France?")

{'messages': [HumanMessage(content='Let me tell you about Jonny. He is a lovely boy with bright blue eyes and curly red hair. He loves playing guitar and eating pizza.', additional_kwargs={}, response_metadata={}, id='18714510-602c-48cb-9194-8741467a1614'),
  AIMessage(content="That sounds like a charming young person! It's great that he has found interests that bring him joy, such as music and food.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 74, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama/llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-72', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cade9-6c32-76c3-bd0c-f71c63d6868d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 28, 'total_tokens': 102, 'input_token_details': {}, 'output_token_details': {}})

In [7]:
# Turn 4: Another filler
print("=== Turn 4: Filler ===")
chat("How many continents are there on Earth?")

{'messages': [HumanMessage(content='Let me tell you about Jonny. He is a lovely boy with bright blue eyes and curly red hair. He loves playing guitar and eating pizza.', additional_kwargs={}, response_metadata={}, id='18714510-602c-48cb-9194-8741467a1614'),
  AIMessage(content="That sounds like a charming young person! It's great that he has found interests that bring him joy, such as music and food.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 74, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama/llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-72', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cade9-6c32-76c3-bd0c-f71c63d6868d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 28, 'total_tokens': 102, 'input_token_details': {}, 'output_token_details': {}})

In [11]:
# Turn 5: Ask about Jonny again - should NOT remember!
# By now: 9 msgs in state, LLM sees only the last 5.
# H1 (Jonny story), A1, H2, A2 are all outside the window.
print("=== Turn 5: Agent should NOT remember Jonny ===")
chat("What color are Jonny's eyes? Do you remember anything about Jonny?")

{'messages': [HumanMessage(content='Let me tell you about Jonny. He is a lovely boy with bright blue eyes and curly red hair. He loves playing guitar and eating pizza.', additional_kwargs={}, response_metadata={}, id='18714510-602c-48cb-9194-8741467a1614'),
  AIMessage(content="That sounds like a charming young person! It's great that he has found interests that bring him joy, such as music and food.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 74, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama/llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-72', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cade9-6c32-76c3-bd0c-f71c63d6868d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 28, 'total_tokens': 102, 'input_token_details': {}, 'output_token_details': {}})

## Inspect PostgreSQL Messages

The full conversation history is always preserved in PostgreSQL,
even though the LLM only sees the last N messages.

In [ ]:
def show_messages(thread_id: str = None):
    """Load the full message history from PostgreSQL and display it."""
    tid = thread_id or THREAD_ID
    config = {"configurable": {"thread_id": tid}}

    with PostgresSaver.from_conn_string(DB_URI) as saver:
        checkpoint_tuple = saver.get_tuple(config)

    if not checkpoint_tuple or not checkpoint_tuple.checkpoint:
        print(f"No checkpoint found for thread '{tid}'")
        return

    messages = checkpoint_tuple.checkpoint.get("channel_values", {}).get("messages", [])
    total = len(messages)
    # max_messages_in_context is 5 in agent.py
    fifo_window = 5

    print(f"Thread: {tid}")
    print(f"Total messages in PostgreSQL: {total}")
    print(f"FIFO window (LLM sees last): {fifo_window}")
    print("=" * 70)

    for i, msg in enumerate(messages):
        in_window = i >= (total - fifo_window)
        marker = ">>" if in_window else "  "
        role = type(msg).__name__.replace("Message", "").lower()
        content = msg.content[:120] + ("..." if len(msg.content) > 120 else "")
        print(f"{marker} [{i + 1:2d}] {role:9s} | {content}")

    print("=" * 70)
    print(">> = inside FIFO window (sent to LLM)")
    print("   = outside window (stored in DB but not visible to LLM)")


show_messages()

In [ ]:
# Raw PostgreSQL query - see what's stored in the checkpoint tables
import psycopg

with psycopg.connect(DB_URI) as conn:
    with conn.cursor() as cur:
        # Show checkpoint tables
        cur.execute(
            "SELECT table_name FROM information_schema.tables "
            "WHERE table_schema = 'public' ORDER BY table_name"
        )
        print("Checkpoint tables in PostgreSQL:")
        for row in cur.fetchall():
            print(f"  - {row[0]}")

        # Show checkpoint metadata for our thread
        print(f"\nCheckpoints for thread '{THREAD_ID}':")
        cur.execute(
            "SELECT thread_id, checkpoint_id, checkpoint_ns "
            "FROM checkpoints WHERE thread_id = %s ORDER BY checkpoint_id",
            (THREAD_ID,),
        )
        rows = cur.fetchall()
        print(f"  Total checkpoint rows: {len(rows)}")
        for row in rows[-3:]:
            print(f"  thread={row[0]}, checkpoint={row[1][:16]}..., ns={row[2]}")

## Send Custom Messages

Use `chat("your message")` to keep talking on the same thread,
or `chat("msg", thread_id="new-thread")` for a fresh conversation.

In [ ]:
# Try your own message
# chat("What have we been talking about?")

In [ ]:
# Inspect messages after your custom interactions
# show_messages()